# Batch optic flow — many sessions at once

Run the per-ROI optic flow (and, optionally, pupil tracking) across **several sessions that are already
prepared** — each folder already has its **ROI config**, its **`log.json`**, and its **noreflection
video**. This notebook does not define ROIs and does not, by default, remove reflections; it assumes
step 0/1/2 of `single_session_optic_flow.ipynb` were already done per session, and just runs the
tracking over the list.

For each session it: checks readiness → builds `session.json` if missing → `compute_roi_flow` →
(optional) `segment_pupil` → collects a one-row summary.

> **Guards.** `RUN_FULL=False` (default) does a cheap short-window flow per session so you can confirm
> the whole list is wired up before committing to the real run (each full session is ~30 min of decode).
> `GENERATE_NOREFLECTION=False` — flip on only if some session is missing its noreflection video and you
> want this notebook to build it from the original. `RUN_PUPIL` toggles the per-animal pupil fit.

In [ ]:
import sys, json, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_HERE = Path.cwd()
# this notebook lives in common/ (next to the modules); fall back to ../common if run from elsewhere
_COMMON = _HERE if (_HERE / 'compute_roi_flow.py').exists() else (_HERE.parent / 'common')
assert (_COMMON / 'compute_roi_flow.py').exists(), f'cannot locate common/ from {_HERE}'
sys.path.insert(0, str(_COMMON)); sys.path.insert(0, str(_COMMON / 'detectors'))
import remove_reflection as rr
import compute_roi_flow as rflow
import segment_pupil as sp
import session_config as scfg
print('modules loaded')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Option A: list the session folders explicitly.
SESSION_DIRS = [
    '/path/to/MAIN_DIR/MOUSE_ID/SESSION',   # <-- edit / extend this list
]
# Option B: instead of the list above, glob a parent folder (set MAIN_DIR, leave SESSION_DIRS = []):
MAIN_DIR = None          # e.g. '/path/to/MAIN_DIR/MOUSE_ID'
GLOB     = '*'           # subfolder pattern under MAIN_DIR

RUN_FULL              = False   # False = short-window preview per session (writes nothing heavy)
RUN_PUPIL            = True    # also fit the pupil (per-animal tuned; QC it in notebook 0 first)
GENERATE_NOREFLECTION = False   # build a missing noreflection video from the original (guarded)

PREVIEW_LO, PREVIEW_HI = 5000, 5200   # window used when RUN_FULL is False

if MAIN_DIR:
    SESSION_DIRS = sorted(str(p) for p in Path(MAIN_DIR).glob(GLOB) if p.is_dir())
SESSION_DIRS = [Path(s).resolve() for s in SESSION_DIRS]
print(f'{len(SESSION_DIRS)} session(s):')
for s in SESSION_DIRS: print('  ', s)
print(f'\nRUN_FULL={RUN_FULL}  RUN_PUPIL={RUN_PUPIL}  GENERATE_NOREFLECTION={GENERATE_NOREFLECTION}')

## 1 — Readiness check

For each folder, is everything the flow needs present? A session is **ready** when it has `log.json`,
a `roi_config*.json`, and a `*noreflection*.mp4`. Missing items are listed so you can fix them (or turn
on `GENERATE_NOREFLECTION` for the noreflection case).

In [ ]:
def _find(d, *tokens_any, exclude=()):
    for p in sorted(Path(d).glob('*.mp4')):
        nm = p.name.lower()
        if all(t in nm for t in tokens_any) and not any(x in nm for x in exclude):
            return p
    return None

def inspect(d):
    d = Path(d)
    log = (d / 'log.json'); rois = sorted(d.glob('roi_config*.json'))
    noref = _find(d, 'noreflection')
    orig  = _find(d, exclude=('noreflection', 'ui', 'viz'))
    sj = (d / 'session.json')
    miss = []
    if not log.exists(): miss.append('log.json')
    if not rois:         miss.append('roi_config*.json')
    if noref is None:    miss.append('noreflection.mp4')
    return dict(session=d.name, has_log=log.exists(), has_roi=bool(rois),
                has_noref=noref is not None, has_original=orig is not None,
                has_session_json=sj.exists(), missing=','.join(miss), ready=(not miss),
                _dir=str(d), _orig=str(orig) if orig else None, _noref=str(noref) if noref else None,
                _roi=str(rois[0]) if rois else None)

status = pd.DataFrame([inspect(d) for d in SESSION_DIRS])
show = ['session', 'has_log', 'has_roi', 'has_noref', 'has_original', 'has_session_json', 'ready', 'missing']
status[show]

In [ ]:
# OPTIONAL: build any missing noreflection video from the original + the session's eye ROIs (guarded)
if GENERATE_NOREFLECTION:
    for _, r in status.iterrows():
        if r.has_noref or not r.has_roi or not r.has_original:
            continue
        boxes, names = rr._eye_bboxes_from_roi_config(r._dir)
        outp = Path(r._dir) / f'{r.session}_noreflection.mp4'
        print(f'{r.session}: generating {outp.name} from eye ROIs {names} ...')
        if RUN_FULL:
            rr.generate_noreflection_video(r._orig, outp, boxes)
        else:
            print('   (RUN_FULL is False -> skipped the full write; flip RUN_FULL to actually build it)')
    status = pd.DataFrame([inspect(d) for d in SESSION_DIRS])   # refresh
else:
    print('GENERATE_NOREFLECTION is False -> not creating any noreflection videos')

## 2 — Run the flow (and pupil) over the ready sessions

`build_session` first (only if `session.json` is missing), then `compute_roi_flow`, then the pupil fit
if `RUN_PUPIL`. Each session is wrapped so one failure does not stop the batch — its error lands in the
summary row instead.

In [ ]:
def process(row):
    d = row._dir; out = dict(session=row.session)
    t0 = time.time()
    try:
        # 1. session contract
        if not (Path(d) / 'session.json').exists():
            scfg.build_session(row.session, d, write=RUN_FULL, verbose=False)
        # 2. per-ROI optic flow
        if RUN_FULL:
            arr = rflow.run(d, write=True, progress=False)
        else:
            arr = rflow.run(d, lo=PREVIEW_LO, hi=PREVIEW_HI, write=False, progress=False)
        out['paw_mag_med'] = float(np.median(arr['paw']['mag'][PREVIEW_LO:PREVIEW_HI])) \
            if 'paw' in arr else np.nan
        out['n_rois'] = len(arr)
        # 3. pupil (optional)
        if RUN_PUPIL:
            if RUN_FULL:
                res = sp.run(d, write=True, progress=False)
            else:
                res = sp.run(d, lo=PREVIEW_LO, hi=PREVIEW_HI, write=False, progress=False)
            rr_ = res['radius']
            out['pupil_valid_pct'] = float(100 * np.mean(~np.isnan(rr_[PREVIEW_LO:PREVIEW_HI])))
            out['pupil_radius_med'] = float(np.nanmedian(rr_[PREVIEW_LO:PREVIEW_HI]))
        out['ok'] = True; out['error'] = ''
    except Exception as ex:
        out['ok'] = False; out['error'] = f'{type(ex).__name__}: {ex}'
        traceback.print_exc()
    out['secs'] = round(time.time() - t0, 1)
    return out

ready = status[status.ready]
print(f'processing {len(ready)} ready session(s)  (RUN_FULL={RUN_FULL})\n')
results = pd.DataFrame([process(r) for _, r in ready.iterrows()])
results

## 3 — Summary

One row per session. In preview mode the numbers come from the short window; with `RUN_FULL=True` they
are the full session and the `opticflow/*.npy` + `pupil_track.npz` files are written into each folder,
ready for the detectors in `../common/detectors/`.

In [ ]:
if len(results):
    n_ok = int(results.ok.sum())
    print(f'{n_ok}/{len(results)} session(s) processed cleanly'
          + ('' if n_ok == len(results) else '  -- see the error column'))
    cols = [c for c in ('session', 'n_rois', 'paw_mag_med', 'pupil_valid_pct', 'pupil_radius_med',
                        'secs', 'ok', 'error') if c in results.columns]
    display(results[cols])
    if not RUN_FULL:
        print('\\nThis was a PREVIEW (short window). Set RUN_FULL=True to process the whole session '
              'and write the arrays.')
else:
    print('no ready sessions -- fix the missing items in the readiness table above')